# IBM Telco Customer Churn: Statistical Analysis

## Introduction

This notebook applies inferential statistical methods to determine whether the customer characteristics identified during exploratory data analysis are meaningfully associated with churn.

The exploratory analysis identified several potential churn drivers, including contract type, internet service, payment method, paperless billing, senior-citizen status, support-service adoption, tenure, monthly charges, and total charges. This notebook evaluates whether those observed differences are statistically significant and measures the strength of each relationship.

The analysis is designed to support business decision-making rather than merely report statistical test results. Each test includes:

- A clearly defined research question
- Null and alternative hypotheses
- Assumption checks
- An appropriate statistical test
- An effect-size measurement
- A business interpretation
- A distinction between statistical significance and practical significance

Statistical association does not establish causation. Results should therefore be interpreted as evidence of relationships that may support customer-retention strategies, not proof that a specific characteristic directly causes churn.

# Table of Contents

1. [Introduction](#Introduction)
2. [Research Questions](#Research-Questions)
3. [Hypothesis Framework](#Hypothesis-Framework)
4. [Data Validation](#Data-Validation)
5. [Categorical Hypothesis Testing](#Categorical-Hypothesis-Testing)
6. [Research Question 1](#Research-Question-1)
7. [Research Question 2](#Research-Question-2)
8. [Research Question 3](#Research-Question-3)
9. [Research Question 4](#Research-Question-4)
10. [Research Question 5](#Research-Question-5)
11. [Research Question 6](#Research-Question-6)
12. [Summary of Categorical Analyses](#Summary-of-Categorical-Analyses)
13. [Numerical Hypothesis Testing](#Numerical-Hypothesis-Testing)
14. [Research Question 7](#Research-Question-7)
15. [Research Question 8](#Research-Question-8)
16. [Research Question 9](#Research-Question-9)
17. [Summary of Numerical Analyses](#Summary-of-Numerical-Analyses)
18. [Executive Summary](#Executive-Summary)
19. [Final Business Recommendations](#Final-Business-Recommendations)
20. [Project Conclusion](#Project-Conclusion)

## Research Questions

This analysis addresses the following research questions:

### Categorical Variables

1. Is customer churn associated with contract type?
2. Is customer churn associated with internet-service type?
3. Is customer churn associated with payment method?
4. Is customer churn associated with paperless billing?
5. Is customer churn associated with senior-citizen status?
6. Is customer churn associated with the number of support services adopted?

### Numerical Variables

7. Do monthly charges differ significantly between churned and retained customers?
8. Do total charges differ significantly between churned and retained customers?
9. Does customer tenure differ significantly between churned and retained customers?

For categorical variables, chi-square tests of independence will be used. The strength of each association will be evaluated using Cramér's V.

For numerical variables, distributional assumptions will be checked before selecting either Welch's independent-samples t-test or the Mann–Whitney U test. Effect sizes will be reported to evaluate whether statistically significant differences are also meaningful from a business perspective.

## Hypothesis Framework

A significance level of **α = 0.05** is used throughout the analysis.

### Chi-Square Tests

For each categorical variable:

- **Null hypothesis (H₀):** The variable and customer churn are independent.
- **Alternative hypothesis (H₁):** The variable and customer churn are associated.

### Numerical Comparisons

For each numerical variable:

- **Null hypothesis (H₀):** There is no difference in the variable's distribution or average value between churned and retained customers.
- **Alternative hypothesis (H₁):** The variable differs between churned and retained customers.

A p-value below 0.05 provides evidence against the null hypothesis. However, p-values are evaluated together with effect sizes, group differences, sample sizes, and business context.

Imports and Display Settings

In [325]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import (
    chi2_contingency,
    levene,
    mannwhitneyu,
    shapiro,
    ttest_ind,
)

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

ALPHA = 0.05
RANDOM_STATE = 42

Locate and Load the Processed Dataset

In [326]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

candidate_paths = [
    PROJECT_ROOT / "data" / "processed" / "telco_customer_churn_clean.csv",
    PROJECT_ROOT / "data" / "processed" / "telco_churn_clean.csv",
    PROJECT_ROOT / "data" / "processed" / "telco_customer_churn_processed.csv",
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    processed_files = list((PROJECT_ROOT / "data" / "processed").glob("*.csv"))

    if len(processed_files) == 1:
        DATA_PATH = processed_files[0]
    else:
        raise FileNotFoundError(
            "Processed dataset could not be identified. "
            "Update candidate_paths with the exact processed CSV filename."
        )

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Dataset loaded from: C:\Users\musta\telco-churn-portfolio\data\processed\telco_churn_clean.csv
Rows: 7,043
Columns: 21


Standardize Column Names

In [327]:
def to_snake_case(column_name: str) -> str:
    """Convert a column name to lowercase snake_case."""
    cleaned = (
        str(column_name)
        .strip()
        .replace("-", "_")
        .replace(" ", "_")
    )

    result = []

    for index, character in enumerate(cleaned):
        if (
            character.isupper()
            and index > 0
            and cleaned[index - 1].islower()
        ):
            result.append("_")

        result.append(character.lower())

    normalized = "".join(result)

    while "__" in normalized:
        normalized = normalized.replace("__", "_")

    return normalized.strip("_")


df.columns = [to_snake_case(column) for column in df.columns]

print("Standardized columns:")
print(df.columns.tolist())

Standardized columns:
['gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn', 'tenure_group']


Resolve Required Columns

In [328]:
COLUMN_ALIASES = {
    "customer_id": [
        "customer_id",
        "customerid",
        "customer_number",
        "customer_no",
        "id",
    ],
    "churn": ["churn", "churn_status", "is_churned", "churn_value"],
    "contract": ["contract", "contract_type"],
    "internet_service": ["internet_service", "internetservice"],
    "payment_method": ["payment_method", "paymentmethod"],
    "paperless_billing": ["paperless_billing", "paperlessbilling"],
    "senior_citizen": ["senior_citizen", "seniorcitizen"],
    "monthly_charges": ["monthly_charges", "monthlycharges"],
    "total_charges": ["total_charges", "totalcharges"],
    "tenure": ["tenure", "tenure_months"],
    "online_security": ["online_security", "onlinesecurity"],
    "online_backup": ["online_backup", "onlinebackup"],
    "device_protection": ["device_protection", "deviceprotection"],
    "tech_support": ["tech_support", "techsupport"],
}


def resolve_column(
    dataframe: pd.DataFrame,
    aliases: list[str],
    required: bool = True,
) -> str | None:
    """Return the first matching column from a list of aliases."""
    for alias in aliases:
        if alias in dataframe.columns:
            return alias

    if required:
        raise KeyError(
            f"None of the expected columns were found: {aliases}"
        )

    return None


print("Available dataset columns:")
print(df.columns.tolist())

resolved_columns = {}

for field, aliases in COLUMN_ALIASES.items():
    # Customer ID is useful for validation but not required for testing.
    is_required = field != "customer_id"

    resolved_columns[field] = resolve_column(
        dataframe=df,
        aliases=aliases,
        required=is_required,
    )

resolved_columns

Available dataset columns:
['gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn', 'tenure_group']


{'customer_id': None,
 'churn': 'churn',
 'contract': 'contract',
 'internet_service': 'internetservice',
 'payment_method': 'paymentmethod',
 'paperless_billing': 'paperlessbilling',
 'senior_citizen': 'seniorcitizen',
 'monthly_charges': 'monthlycharges',
 'total_charges': 'totalcharges',
 'tenure': 'tenure',
 'online_security': 'onlinesecurity',
 'online_backup': 'onlinebackup',
 'device_protection': 'deviceprotection',
 'tech_support': 'techsupport'}

Rename Columns to Canonical Analysis Names

In [329]:
rename_map = {
    actual_name: canonical_name
    for canonical_name, actual_name in resolved_columns.items()
    if actual_name is not None and actual_name != canonical_name
}

analysis_df = df.rename(columns=rename_map).copy()

# Create a stable internal row identifier if the cleaned dataset
# does not retain the original customer ID.
if resolved_columns["customer_id"] is None:
    analysis_df.insert(
        0,
        "customer_id",
        [f"ROW_{index:05d}" for index in range(1, len(analysis_df) + 1)],
    )

    print(
        "Original customer ID was not found. "
        "A unique internal row identifier was created for validation."
    )

required_columns = [
    "customer_id",
    "churn",
    "contract",
    "internet_service",
    "payment_method",
    "paperless_billing",
    "senior_citizen",
    "monthly_charges",
    "total_charges",
    "tenure",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in analysis_df.columns
]

if missing_required_columns:
    raise KeyError(
        f"Missing required analysis columns: {missing_required_columns}"
    )

print("All required statistical-analysis columns are available.")

Original customer ID was not found. A unique internal row identifier was created for validation.
All required statistical-analysis columns are available.


Validate Numeric Columns

In [330]:
numeric_columns = [
    "tenure",
    "monthly_charges",
    "total_charges",
]

for column in numeric_columns:
    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="coerce",
    )

numeric_validation = pd.DataFrame(
    {
        "data_type": analysis_df[numeric_columns].dtypes.astype(str),
        "missing_values": analysis_df[numeric_columns].isna().sum(),
        "minimum": analysis_df[numeric_columns].min(),
        "maximum": analysis_df[numeric_columns].max(),
    }
)

display(numeric_validation)

,data_type,missing_values,minimum,maximum
tenure,int64,0,0.0000,72.0000
monthly_charges,float64,0,18.2500,118.7500
total_charges,float64,0,0.0000,"8,684.8000"


Standardize Churn

In [331]:
analysis_df["churn"] = (
    analysis_df["churn"]
    .astype(str)
    .str.strip()
    .str.lower()
)

churn_mapping = {
    "yes": "Churned",
    "no": "Retained",
    "1": "Churned",
    "0": "Retained",
    "true": "Churned",
    "false": "Retained",
    "churned": "Churned",
    "retained": "Retained",
}

analysis_df["churn_status"] = analysis_df["churn"].map(churn_mapping)

unmapped_churn_values = analysis_df.loc[
    analysis_df["churn_status"].isna(),
    "churn",
].unique()

if len(unmapped_churn_values) > 0:
    raise ValueError(
        f"Unrecognized churn values: {unmapped_churn_values}"
    )

analysis_df["churn_flag"] = (
    analysis_df["churn_status"] == "Churned"
).astype(int)

analysis_df["churn_status"].value_counts()

churn_status
Retained    5174
Churned     1869
Name: count, dtype: int64

Create the Support-Service Count

In [332]:
support_service_columns = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
]

analysis_df["has_internet"] = (
    analysis_df["internet_service"]
    .astype(str)
    .str.strip()
    .str.lower()
    != "no"
)

for col in support_service_columns:
    analysis_df[col] = pd.to_numeric(
        analysis_df[col],
        errors="coerce"
    ).fillna(0).astype(int)

analysis_df["support_services_count"] = (
    analysis_df[support_service_columns]
    .sum(axis=1)
)

analysis_df.loc[
    ~analysis_df["has_internet"],
    "support_services_count"
] = np.nan

analysis_df["support_services_count"] = (
    analysis_df["support_services_count"]
    .astype("Int64")
)

analysis_df[
    [
        "internet_service",
        *support_service_columns,
        "support_services_count"
    ]
].head()

,internet_service,online_security,online_backup,device_protection,tech_support,support_services_count
0,DSL,0,1,0,0,1
1,DSL,1,0,1,0,2
2,DSL,1,1,0,0,2
3,DSL,1,0,1,1,3
4,Fiber optic,0,0,0,0,0


Core Dataset Validation

In [333]:
validation_summary = pd.Series(
    {
        "rows": len(analysis_df),
        "unique_customers": analysis_df["customer_id"].nunique(),
        "duplicate_customer_ids": analysis_df["customer_id"].duplicated().sum(),
        "churned_customers": int(analysis_df["churn_flag"].sum()),
        "retained_customers": int((analysis_df["churn_flag"] == 0).sum()),
        "overall_churn_rate": analysis_df["churn_flag"].mean(),
        "missing_tenure": analysis_df["tenure"].isna().sum(),
        "missing_monthly_charges": analysis_df["monthly_charges"].isna().sum(),
        "missing_total_charges": analysis_df["total_charges"].isna().sum(),
    }
)

display(validation_summary.to_frame("value"))

,value
rows,"7,043.0000"
unique_customers,"7,043.0000"
duplicate_customer_ids,0.0000
churned_customers,"1,869.0000"
retained_customers,"5,174.0000"
overall_churn_rate,0.2654
missing_tenure,0.0000
missing_monthly_charges,0.0000
missing_total_charges,0.0000


Validate Canonical Numerical Results

In [334]:
canonical_numeric_validation = (
    analysis_df
    .groupby("churn_status", observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        average_tenure=("tenure", "mean"),
        average_monthly_charges=("monthly_charges", "mean"),
        average_total_charges=("total_charges", "mean"),
        monthly_charge_exposure=("monthly_charges", "sum"),
    )
    .reindex(["Retained", "Churned"])
)

display(canonical_numeric_validation)

,customers,average_tenure,average_monthly_charges,average_total_charges,monthly_charge_exposure
churn_status,,,,,
Retained,5174,37.5700,61.2651,"2,549.9114","316,985.7500"
Churned,1869,17.9791,74.4413,"1,531.7961","139,130.8500"


Validate Corrected Support-Service Results

In [335]:
support_service_validation = (
    analysis_df.loc[analysis_df["has_internet"]]
    .groupby("support_services_count", observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum"),
        churn_rate=("churn_flag", "mean"),
    )
    .reset_index()
)

support_service_validation["churn_rate_pct"] = (
    support_service_validation["churn_rate"] * 100
)

display(support_service_validation)

,support_services_count,customers,churned_customers,churn_rate,churn_rate_pct
0,0,1267,718,0.5667,56.6693
1,1,1467,570,0.3885,38.8548
2,2,1372,326,0.2376,23.7609
3,3,941,117,0.1243,12.4336
4,4,470,25,0.0532,5.3191


## Data Validation Conclusion

The statistical-analysis dataset contains 7,043 unique customer records, including 1,869 churned customers and 5,174 retained customers. The resulting overall churn rate is 26.54%, matching the validated SQL analysis and exploratory data analysis (EDA) notebook.

Key numerical metrics—including average tenure, monthly charges, total charges, and monthly charge exposure—also align with the project's canonical business results, confirming consistency across all stages of the analysis pipeline.

The processed analytical dataset encodes support-service adoption as binary indicators (0 = not adopted, 1 = adopted). Customers without internet service are excluded from the support-service analysis because they are not eligible to subscribe to internet-based support services. This treatment is consistent with the SQL implementation and prevents customers from being incorrectly classified as having adopted zero support services.

Using this validated approach, customer churn decreases consistently as support-service adoption increases, from **56.67%** among customers with no support services to **5.32%** among customers utilizing all four available support services. This relationship is one of the strongest patterns identified during exploratory analysis and will be formally evaluated using inferential statistical methods in the following sections.

The dataset has been successfully validated and is suitable for statistical hypothesis testing.

# Statistical Analysis

## Methodology

The objective of this section is to determine whether the relationships observed during exploratory data analysis are statistically significant.

Each hypothesis test follows a consistent analytical workflow:

1. State the research question.
2. Define the null and alternative hypotheses.
3. Select an appropriate statistical test.
4. Verify test assumptions where applicable.
5. Calculate the test statistic and p-value.
6. Measure the magnitude of the relationship using an effect-size metric.
7. Interpret the findings from both statistical and business perspectives.

Unless otherwise stated, statistical significance is evaluated using a significance level of **α = 0.05**.

Because the IBM Telco Customer Churn dataset contains more than 7,000 customer records, statistically significant results should also be evaluated using effect sizes to determine whether the observed relationships are practically meaningful for business decision-making.

Effect Size Functions

In [336]:
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency

In [337]:
def cramers_v(contingency_table):
    """
    Calculate Cramer's V for a chi-square test of independence.
    """

    chi2 = chi2_contingency(contingency_table)[0]

    n = contingency_table.to_numpy().sum()

    r, c = contingency_table.shape

    return np.sqrt(chi2 / (n * (min(r - 1, c - 1))))

Interpretation Function

In [338]:
def interpret_cramers_v(v):
    """
    Interpret the magnitude of Cramer's V.
    """

    if v < 0.10:
        return "Negligible"

    elif v < 0.30:
        return "Weak"

    elif v < 0.50:
        return "Moderate"

    else:
        return "Strong"

Chi-Square Helper

In [339]:
def chi_square_analysis(df, feature, target="churn_status"):
    """
    Perform a chi-square test and return a summary.
    """

    contingency = pd.crosstab(df[feature], df[target])

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    cramers = cramers_v(contingency)

    return {
        "Contingency Table": contingency,
        "Chi-Square": chi2,
        "Degrees of Freedom": dof,
        "P-Value": p_value,
        "Cramers V": cramers,
        "Effect": interpret_cramers_v(cramers),
    }

Section 2 — Contract Type

## Research Question 1

### Is customer churn associated with contract type?

Contract duration is expected to influence customer retention because longer agreements increase switching costs and typically indicate greater customer commitment.

### Hypotheses

**Null hypothesis (H₀)**

Customer churn is independent of contract type.

**Alternative hypothesis (H₁)**

Customer churn is associated with contract type.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramér's V

In [340]:
contract_results = chi_square_analysis(
    analysis_df,
    "contract"
)

contract_results["Contingency Table"]

churn_status,Churned,Retained
contract,,
Month-to-month,1655,2220
One year,166,1307
Two year,48,1647


In [341]:
summary = pd.DataFrame(
    {
        "Statistic": [
            "Chi-Square",
            "Degrees of Freedom",
            "P-Value",
            "Cramer's V",
            "Association Strength",
        ],
        "Value": [
            contract_results["Chi-Square"],
            contract_results["Degrees of Freedom"],
            contract_results["P-Value"],
            contract_results["Cramers V"],
            contract_results["Effect"],
        ],
    }
)

summary

,Statistic,Value
0,Chi-Square,"1,184.5966"
1,Degrees of Freedom,2
2,P-Value,0.0000
3,Cramer's V,0.4101
4,Association Strength,Moderate


### Business Interpretation

The chi-square test evaluates whether customer churn and contract type are statistically independent.

If the p-value is less than 0.05, the null hypothesis is rejected, indicating that contract type is significantly associated with customer churn.

The effect size, measured using Cramér's V, indicates the practical strength of the relationship. While large datasets often produce statistically significant p-values, Cramér's V helps determine whether the relationship is meaningful from a business perspective.

The exploratory analysis showed that customers on month-to-month contracts experienced substantially higher churn than customers with one-year or two-year contracts. Statistical testing will determine whether this observed pattern is unlikely to have occurred by chance.

If confirmed, contract type represents one of the strongest opportunities for customer-retention initiatives, including contract-renewal incentives, loyalty programs, and targeted outreach to customers approaching renewal periods.

## Statistical Assumptions

Each statistical test relies on specific assumptions to ensure valid interpretation.

### Chi-Square Test of Independence

The Chi-Square Test assumes:

- Observations are independent.
- Categories are mutually exclusive.
- Expected frequencies are sufficiently large (generally at least 5 in each cell).

Expected frequencies will be reviewed for each contingency table to verify the appropriateness of the test.

### Independent-Samples Tests

For numerical variables, the following assumptions will be evaluated before selecting a statistical test:

- Independence of observations
- Approximate normality of each group
- Homogeneity of variances

If these assumptions are not satisfied, the non-parametric Mann–Whitney U test will be used instead of Welch's t-test.

### Decision Rule

For all analyses:

- If p < 0.05, reject the null hypothesis.
- If p ≥ 0.05, fail to reject the null hypothesis.

Rejecting the null hypothesis indicates evidence of an association or difference within this dataset. It does not establish a causal relationship.

Expected Frequencies Check

In [342]:
def chi_square_analysis(df, feature, target="churn_status"):
    """
    Perform a Chi-Square Test of Independence and return
    all results needed for reporting.
    """

    contingency = pd.crosstab(df[feature], df[target])

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    expected_df = pd.DataFrame(
        expected,
        index=contingency.index,
        columns=contingency.columns,
    )

    cramers = cramers_v(contingency)

    return {
        "contingency": contingency,
        "expected": expected_df,
        "chi_square": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value,
        "cramers_v": cramers,
        "effect": interpret_cramers_v(cramers),
        "assumption_passed": expected_df.min().min() >= 5,
    }

In [343]:
contract_results = chi_square_analysis(
    analysis_df,
    "contract"
)

display(contract_results["contingency"])

churn_status,Churned,Retained
contract,,
Month-to-month,1655,2220
One year,166,1307
Two year,48,1647


Display the expected frequencies

In [344]:
display(contract_results["expected"])

churn_status,Churned,Retained
contract,,
Month-to-month,"1,028.3082","2,846.6918"
One year,390.8898,"1,082.1102"
Two year,449.8019,"1,245.1981"


Verify the assumption

In [345]:
print(
    "Minimum expected frequency:",
    contract_results["expected"].min().min()
)

print(
    "Chi-Square assumptions satisfied:",
    contract_results["assumption_passed"]
)

Minimum expected frequency: 390.889819679114
Chi-Square assumptions satisfied: True


Contract Type (Research Question 1)

In [346]:
contract_results = chi_square_analysis(
    analysis_df,
    "contract"
)

In [347]:
display(contract_results["contingency"])
display(contract_results["expected"])

churn_status,Churned,Retained
contract,,
Month-to-month,1655,2220
One year,166,1307
Two year,48,1647


churn_status,Churned,Retained
contract,,
Month-to-month,"1,028.3082","2,846.6918"
One year,390.8898,"1,082.1102"
Two year,449.8019,"1,245.1981"


In [348]:
summary = pd.DataFrame({
    "Statistic": [
        "Chi-Square",
        "Degrees of Freedom",
        "P-Value",
        "Cramer's V",
        "Association Strength",
        "Assumptions Satisfied",
    ],
    "Value": [
        round(contract_results["chi_square"], 3),
        contract_results["degrees_of_freedom"],
        f"{contract_results['p_value']:.4e}",
        round(contract_results["cramers_v"], 3),
        contract_results["effect"],
        contract_results["assumption_passed"],
    ],
})

display(summary)

,Statistic,Value
0,Chi-Square,"1,184.5970"
1,Degrees of Freedom,2
2,P-Value,5.8630e-258
3,Cramer's V,0.4100
4,Association Strength,Moderate
5,Assumptions Satisfied,True


### Statistical Interpretation

The Chi-Square Test of Independence identified a statistically significant association between customer contract type and churn status (χ² = 1,184.60, df = 2, p < 0.001). Therefore, the null hypothesis is rejected, providing strong evidence that customer churn is associated with contract type.

All expected cell frequencies exceeded the recommended minimum threshold of five, confirming that the assumptions of the Chi-Square Test of Independence were satisfied and that the test results are statistically valid.

The effect size, measured using Cramér's V (V = 0.41), indicates a **moderate association** between contract type and churn. This suggests that contract type is not only statistically significant but also practically meaningful, making it an important factor when developing customer retention strategies.

These statistical findings validate the patterns observed during exploratory data analysis, where customers on month-to-month contracts exhibited substantially higher churn rates than customers with one-year and two-year contracts.

### Business Insight

Contract type represents one of the strongest customer characteristics associated with churn in this analysis. The moderate effect size indicates that the relationship is meaningful from both a statistical and business perspective, rather than being solely a consequence of the large sample size.

Customers on month-to-month contracts experience considerably higher churn rates than customers with longer-term agreements, suggesting that contractual commitment is an important indicator of customer retention. Organizations should prioritize proactive retention initiatives for month-to-month customers, including renewal incentives, personalized offers, bundled services, and loyalty programs designed to encourage longer-term commitments.

Although contract type is strongly associated with churn, it should not be interpreted as the sole cause of customer attrition. Customer retention strategies are likely to be most effective when contract information is combined with other high-risk characteristics, such as tenure, internet service type, support-service adoption, and payment method.

## Research Question 2

### Is customer churn associated with internet service type?

Internet service represents one of the organization's primary products and may influence customer satisfaction, service quality, and customer retention. This analysis evaluates whether customer churn is associated with the type of internet service subscribed to by the customer.

### Hypotheses

**Null Hypothesis (H₀)**

Customer churn is independent of internet service type.

**Alternative Hypothesis (H₁)**

Customer churn is associated with internet service type.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramér's V

In [349]:
internet_results = chi_square_analysis(
    analysis_df,
    "internet_service"
)

In [350]:
display(internet_results["contingency"])

display(internet_results["expected"])

churn_status,Churned,Retained
internet_service,,
DSL,459,1962
Fiber optic,1297,1799
No,113,1413


churn_status,Churned,Retained
internet_service,,
DSL,642.4605,"1,778.5395"
Fiber optic,821.5851,"2,274.4149"
No,404.9544,"1,121.0456"


In [351]:
summary = pd.DataFrame({
    "Statistic": [
        "Chi-Square",
        "Degrees of Freedom",
        "P-Value",
        "Cramer's V",
        "Association Strength",
        "Assumptions Satisfied",
    ],
    "Value": [
        round(internet_results["chi_square"], 3),
        internet_results["degrees_of_freedom"],
        "< 0.001" if internet_results["p_value"] < 0.001 else round(internet_results["p_value"], 4),
        round(internet_results["cramers_v"], 3),
        internet_results["effect"],
        internet_results["assumption_passed"],
    ],
})

display(summary)

,Statistic,Value
0,Chi-Square,732.3100
1,Degrees of Freedom,2
2,P-Value,< 0.001
3,Cramer's V,0.3220
4,Association Strength,Moderate
5,Assumptions Satisfied,True


### Statistical Interpretation

The Chi-Square Test of Independence identified a statistically significant association between internet service type and customer churn (χ² = 732.31, df = 2, p < 0.001). Therefore, the null hypothesis is rejected, indicating that customer churn is associated with the type of internet service subscribed to by the customer.

All expected cell frequencies exceeded the recommended minimum threshold of five, confirming that the assumptions of the Chi-Square Test of Independence were satisfied and that the test results are statistically valid.

The effect size, measured using Cramér's V (V = 0.322), indicates a **moderate association** between internet service type and customer churn. This suggests that internet service type is not only statistically significant but also practically meaningful when evaluating customer retention risk.

These findings validate the patterns identified during exploratory data analysis, where customers with fiber optic internet exhibited substantially higher churn rates than customers using DSL or customers without internet service.

### Business Insight

Internet service type is an important characteristic associated with customer churn. Customers subscribed to fiber optic internet experienced the highest churn rates in the dataset, while customers with DSL or no internet service demonstrated considerably lower churn rates.

Although this analysis does not identify the underlying causes of the observed relationship, it highlights the fiber optic customer segment as a priority for further investigation. Factors such as pricing, service quality, competitive alternatives, customer expectations, or support experience may contribute to the elevated churn observed within this group.

Organizations should prioritize retention initiatives for high-risk fiber optic customers while conducting additional analyses to identify the operational or customer experience factors driving attrition.

In [352]:
payment_results = chi_square_analysis(
    analysis_df,
    "payment_method"
)

In [353]:
display(payment_results["contingency"])

churn_status,Churned,Retained
payment_method,,
Bank transfer (automatic),258,1286
Credit card (automatic),232,1290
Electronic check,1071,1294
Mailed check,308,1304


In [354]:
display(payment_results["expected"])

print(
    "Minimum expected frequency:",
    payment_results["expected"].min().min()
)

print(
    "Chi-Square assumptions satisfied:",
    payment_results["assumption_passed"]
)

churn_status,Churned,Retained
payment_method,,
Bank transfer (automatic),409.7311,"1,134.2689"
Credit card (automatic),403.8929,"1,118.1071"
Electronic check,627.5997,"1,737.4003"
Mailed check,427.7762,"1,184.2238"


Minimum expected frequency: 403.89294334800513
Chi-Square assumptions satisfied: True


In [355]:
summary = pd.DataFrame({
    "Statistic": [
        "Chi-Square",
        "Degrees of Freedom",
        "P-Value",
        "Cramer's V",
        "Association Strength",
        "Assumptions Satisfied",
    ],
    "Value": [
        round(payment_results["chi_square"], 3),
        payment_results["degrees_of_freedom"],
        "< 0.001" if payment_results["p_value"] < 0.001 else round(payment_results["p_value"], 4),
        round(payment_results["cramers_v"], 3),
        payment_results["effect"],
        payment_results["assumption_passed"],
    ],
})

display(summary)

,Statistic,Value
0,Chi-Square,648.1420
1,Degrees of Freedom,3
2,P-Value,< 0.001
3,Cramer's V,0.3030
4,Association Strength,Moderate
5,Assumptions Satisfied,True


### Statistical Interpretation

The Chi-Square Test of Independence identified a statistically significant association between payment method and customer churn (χ² = 648.14, df = 3, p < 0.001). Therefore, the null hypothesis is rejected, indicating that customer churn is associated with customers' preferred payment method.

All expected cell frequencies exceeded the recommended minimum threshold of five, confirming that the assumptions of the Chi-Square Test of Independence were satisfied and that the test results are statistically valid.

The effect size, measured using Cramér's V (V = 0.303), indicates a **moderate association** between payment method and customer churn. This suggests that payment method is not only statistically significant but also practically meaningful when assessing customer retention risk.

These findings support the exploratory data analysis, which showed that customers using electronic checks experienced substantially higher churn rates than customers using bank transfers, credit cards, or mailed checks.

### Business Insight

Payment method is an important characteristic associated with customer churn. Customers using electronic checks exhibited the highest churn rates in the dataset, while customers using automatic payment methods, including bank transfers and credit cards, demonstrated substantially lower churn rates.

Although this analysis does not establish a causal relationship, it suggests that payment behavior may reflect broader differences in customer engagement, payment convenience, or account management preferences.

Organizations should further investigate the factors contributing to elevated churn among electronic-check customers and consider initiatives that encourage enrollment in automated payment methods. Improving billing convenience and promoting automatic payment options may contribute to improved customer retention when combined with broader retention strategies.

## Research Question 4

### Is customer churn associated with paperless billing?

Paperless billing represents a customer's preferred billing and communication method and may reflect differences in customer engagement, account management, or payment behavior. This analysis evaluates whether customer churn is associated with paperless billing enrollment.

### Hypotheses

**Null Hypothesis (H₀)**

Customer churn is independent of paperless billing enrollment.

**Alternative Hypothesis (H₁)**

Customer churn is associated with paperless billing enrollment.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramér's V

In [356]:
paperless_results = chi_square_analysis(
    analysis_df,
    "paperless_billing"
)

In [357]:
display(paperless_results["contingency"])

churn_status,Churned,Retained
paperless_billing,,
No,469,2403
Yes,1400,2771


In [358]:
display(paperless_results["expected"])

print(
    "Minimum expected frequency:",
    paperless_results["expected"].min().min()
)

print(
    "Chi-Square assumptions satisfied:",
    paperless_results["assumption_passed"]
)

churn_status,Churned,Retained
paperless_billing,,
No,762.1423,"2,109.8577"
Yes,"1,106.8577","3,064.1423"


Minimum expected frequency: 762.1422689194945
Chi-Square assumptions satisfied: True


In [359]:
summary = pd.DataFrame({
    "Statistic": [
        "Chi-Square",
        "Degrees of Freedom",
        "P-Value",
        "Cramer's V",
        "Association Strength",
        "Assumptions Satisfied",
    ],
    "Value": [
        round(paperless_results["chi_square"], 3),
        paperless_results["degrees_of_freedom"],
        "< 0.001" if paperless_results["p_value"] < 0.001 else round(paperless_results["p_value"], 4),
        round(paperless_results["cramers_v"], 3),
        paperless_results["effect"],
        paperless_results["assumption_passed"],
    ],
})

display(summary)

,Statistic,Value
0,Chi-Square,258.2780
1,Degrees of Freedom,1
2,P-Value,< 0.001
3,Cramer's V,0.1910
4,Association Strength,Weak
5,Assumptions Satisfied,True


### Statistical Interpretation

The Chi-Square Test of Independence identified a statistically significant association between paperless billing enrollment and customer churn (χ² = 258.28, df = 1, p < 0.001). Therefore, the null hypothesis is rejected, indicating that customer churn is associated with paperless billing enrollment.

All expected cell frequencies exceeded the recommended minimum threshold of five, confirming that the assumptions of the Chi-Square Test of Independence were satisfied and that the test results are statistically valid.

The effect size, measured using Cramér's V (V = 0.191), indicates a **weak association** between paperless billing enrollment and customer churn. Although the relationship is statistically significant, its practical impact is smaller than the associations observed for contract type, internet service type, and payment method.

These findings support the exploratory data analysis, which showed that customers enrolled in paperless billing experienced higher churn rates than customers using traditional paper billing.

### Business Insight

Paperless billing is associated with customer churn; however, the relationship is relatively weak compared with other customer characteristics examined in this analysis. This suggests that paperless billing alone is unlikely to be a primary driver of customer attrition.

Instead, paperless billing should be viewed as one component of a broader customer risk profile. Customers enrolled in paperless billing may warrant additional attention when combined with stronger risk factors such as month-to-month contracts, fiber optic internet service, or limited support-service adoption.

Organizations should avoid designing retention strategies based solely on paperless billing status and instead incorporate it into multi-factor customer risk models that consider the combined influence of multiple customer characteristics.

## Research Question 5

### Is customer churn associated with senior-citizen status?

Customer age group may influence service needs, communication preferences, and customer retention. This analysis evaluates whether customer churn is associated with senior-citizen status.

### Hypotheses

**Null Hypothesis (H₀)**

Customer churn is independent of senior-citizen status.

**Alternative Hypothesis (H₁)**

Customer churn is associated with senior-citizen status.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramér's V

In [360]:
senior_results = chi_square_analysis(
    analysis_df,
    "senior_citizen"
)

In [361]:
display(senior_results["contingency"])

churn_status,Churned,Retained
senior_citizen,,
0,1393,4508
1,476,666


In [362]:
display(senior_results["expected"])

print(
    "Minimum expected frequency:",
    senior_results["expected"].min().min()
)

print(
    "Chi-Square assumptions satisfied:",
    senior_results["assumption_passed"]
)

churn_status,Churned,Retained
senior_citizen,,
0,"1,565.9476","4,335.0524"
1,303.0524,838.9476


Minimum expected frequency: 303.0523924464007
Chi-Square assumptions satisfied: True


In [363]:
summary = pd.DataFrame({
    "Statistic": [
        "Chi-Square",
        "Degrees of Freedom",
        "P-Value",
        "Cramer's V",
        "Association Strength",
        "Assumptions Satisfied",
    ],
    "Value": [
        round(senior_results["chi_square"], 3),
        senior_results["degrees_of_freedom"],
        "< 0.001" if senior_results["p_value"] < 0.001 else round(senior_results["p_value"], 4),
        round(senior_results["cramers_v"], 3),
        senior_results["effect"],
        senior_results["assumption_passed"],
    ],
})

display(summary)

,Statistic,Value
0,Chi-Square,159.4260
1,Degrees of Freedom,1
2,P-Value,< 0.001
3,Cramer's V,0.1500
4,Association Strength,Weak
5,Assumptions Satisfied,True


### Statistical Interpretation

The Chi-Square Test of Independence identified a statistically significant association between senior-citizen status and customer churn (χ² = 159.43, df = 1, p < 0.001). Therefore, the null hypothesis is rejected, indicating that customer churn is associated with senior-citizen status.

All expected cell frequencies exceeded the recommended minimum threshold of five, confirming that the assumptions of the Chi-Square Test of Independence were satisfied and that the test results are statistically valid.

The effect size, measured using Cramér's V (V = 0.150), indicates a **weak association** between senior-citizen status and customer churn. Although the relationship is statistically significant, its practical influence is smaller than those observed for contract type, internet service type, and payment method.

These findings validate the exploratory data analysis, which showed that senior citizens experienced higher churn rates than non-senior customers.

### Business Insight

Senior-citizen status is associated with customer churn; however, the relatively weak effect size suggests that age group alone is not a primary determinant of customer attrition.

The elevated churn observed among senior customers may reflect differences in customer needs, service expectations, pricing sensitivity, or support requirements. Organizations should therefore avoid designing retention initiatives based solely on age group and instead combine senior-citizen status with stronger predictors such as contract type, internet service type, payment method, and support-service adoption.

In practice, senior-citizen status can improve customer risk segmentation when incorporated into a broader, multi-factor retention strategy rather than being used as an independent indicator of churn risk.

## Research Question 6

### Is customer churn associated with support-service adoption?

Support services, including online security, online backup, device protection, and technical support, are intended to increase the value customers receive from their subscription. This analysis evaluates whether customer churn is associated with the number of support services adopted by customers with internet service.

Customers without internet service are excluded from this analysis because they are not eligible to subscribe to internet-based support services.

### Hypotheses

**Null Hypothesis (H₀)**

Customer churn is independent of support-service adoption.

**Alternative Hypothesis (H₁)**

Customer churn is associated with support-service adoption.

### Statistical Test

Chi-Square Test of Independence

### Effect Size

Cramér's V

In [364]:
support_results = chi_square_analysis(
    analysis_df[analysis_df["has_internet"]],
    "support_services_count"
)

In [365]:
display(support_results["contingency"])

churn_status,Churned,Retained
support_services_count,,
0,718,549
1,570,897
2,326,1046
3,117,824
4,25,445


In [366]:
display(support_results["expected"])

print(
    "Minimum expected frequency:",
    support_results["expected"].min().min()
)

print(
    "Chi-Square assumptions satisfied:",
    support_results["assumption_passed"]
)

churn_status,Churned,Retained
support_services_count,,
0,403.2721,863.7279
1,466.9299,"1,000.0701"
2,436.6924,935.3076
3,299.5099,641.4901
4,149.5958,320.4042


Minimum expected frequency: 149.59579481602321
Chi-Square assumptions satisfied: True


In [367]:
summary = pd.DataFrame({
    "Statistic": [
        "Chi-Square",
        "Degrees of Freedom",
        "P-Value",
        "Cramer's V",
        "Association Strength",
        "Assumptions Satisfied",
    ],
    "Value": [
        round(support_results["chi_square"], 3),
        support_results["degrees_of_freedom"],
        "< 0.001" if support_results["p_value"] < 0.001 else round(support_results["p_value"], 4),
        round(support_results["cramers_v"], 3),
        support_results["effect"],
        support_results["assumption_passed"],
    ],
})

display(summary)

,Statistic,Value
0,Chi-Square,750.2050
1,Degrees of Freedom,4
2,P-Value,< 0.001
3,Cramer's V,0.3690
4,Association Strength,Moderate
5,Assumptions Satisfied,True


### Statistical Interpretation

The Chi-Square Test of Independence identified a statistically significant association between support-service adoption and customer churn (χ² = 750.21, df = 4, p < 0.001). Therefore, the null hypothesis is rejected, indicating that customer churn is associated with the number of support services adopted by customers.

All expected cell frequencies exceeded the recommended minimum threshold of five, confirming that the assumptions of the Chi-Square Test of Independence were satisfied and that the test results are statistically valid.

The effect size, measured using Cramér's V (V = 0.369), indicates a **moderate association** between support-service adoption and customer churn. This represents one of the strongest relationships identified in the categorical analyses and suggests that support-service adoption is both statistically significant and practically meaningful.

These findings validate the exploratory data analysis, which demonstrated a consistent decline in churn rates as customers adopted additional support services, decreasing from 56.67% among customers with no support services to 5.32% among customers utilizing all four measured services.

### Business Insight

Support-service adoption represents one of the strongest customer behaviors associated with retention in this analysis. Customers who utilize a greater number of support services consistently experience substantially lower churn rates than customers with limited or no support-service adoption.

Unlike demographic characteristics, support-service adoption is a customer behavior that organizations can actively influence through product education, onboarding programs, bundled service offerings, and targeted customer engagement initiatives. Increasing customer awareness and adoption of services such as online security, online backup, device protection, and technical support may strengthen customer engagement and reduce churn.

These findings suggest that encouraging support-service adoption should be considered a strategic retention initiative, particularly for high-risk customers identified through other factors such as month-to-month contracts or fiber optic internet service.

# Summary of Categorical Analyses

The six Chi-Square tests evaluated whether key customer characteristics were associated with customer churn. While all variables demonstrated statistically significant relationships with churn (p < 0.001), the strength of these associations varied considerably.

To support business decision-making, the results are summarized below by combining statistical significance, effect size, observed churn rates from the exploratory data analysis, and the practical relevance of each variable. This comparison highlights which customer characteristics represent the most meaningful opportunities for customer retention initiatives.

In [368]:
categorical_summary = pd.DataFrame({
    "Rank": [1, 2, 3, 4, 5, 6],
    "Variable": [
        "Contract Type",
        "Support Services",
        "Internet Service",
        "Payment Method",
        "Paperless Billing",
        "Senior Citizen"
    ],
    "Highest-Risk Group": [
        "Month-to-month",
        "0 Support Services",
        "Fiber Optic",
        "Electronic Check",
        "Paperless Billing",
        "Senior"
    ],
    "Highest Churn Rate": [
        "42.71%",
        "56.67%",
        "41.89%",
        "45.29%",
        "33.57%",
        "41.68%"
    ],
    "Chi-Square": [
        1184.597,
        750.205,
        732.310,
        648.142,
        258.278,
        159.426
    ],
    "Cramer's V": [
        0.410,
        0.369,
        0.322,
        0.303,
        0.191,
        0.150
    ],
    "Association": [
        "Moderate",
        "Moderate",
        "Moderate",
        "Moderate",
        "Weak",
        "Weak"
    ],
    "Business Priority": [
        "Very High",
        "Very High",
        "High",
        "High",
        "Medium",
        "Medium"
    ]
})

display(categorical_summary)

,Rank,Variable,Highest-Risk Group,Highest Churn Rate,Chi-Square,Cramer's V,Association,Business Priority
0,1,Contract Type,Month-to-month,42.71%,"1,184.5970",0.4100,Moderate,Very High
1,2,Support Services,0 Support Services,56.67%,750.2050,0.3690,Moderate,Very High
2,3,Internet Service,Fiber Optic,41.89%,732.3100,0.3220,Moderate,High
3,4,Payment Method,Electronic Check,45.29%,648.1420,0.3030,Moderate,High
4,5,Paperless Billing,Paperless Billing,33.57%,258.2780,0.1910,Weak,Medium
5,6,Senior Citizen,Senior,41.68%,159.4260,0.1500,Weak,Medium


## Key Findings

The statistical analyses identified **contract type** as the strongest categorical factor associated with customer churn, followed closely by **support-service adoption**, **internet service type**, and **payment method**. Each of these variables demonstrated moderate effect sizes, indicating meaningful practical relationships with customer churn beyond statistical significance.

Support-service adoption emerged as one of the most actionable findings. Customers without support services experienced the highest observed churn rate (56.67%), while customers using all four measured support services exhibited the lowest churn rate (5.32%). This consistent pattern suggests that increasing support-service adoption may represent an effective customer retention strategy.

In contrast, **paperless billing** and **senior-citizen status** demonstrated statistically significant but comparatively weaker associations with churn. These variables may improve customer risk segmentation when combined with stronger operational factors but are less suitable as primary drivers of retention initiatives.

## Business Recommendations

Based on the categorical analyses, the following business actions are recommended:

- Prioritize retention programs for customers on month-to-month contracts, as contract type exhibited the strongest association with churn.
- Increase customer adoption of support services through onboarding, education, bundled offerings, and targeted promotions.
- Investigate the elevated churn observed among fiber optic customers to identify potential pricing, service quality, or competitive factors.
- Encourage automatic payment methods among customers currently using electronic checks to improve customer retention.
- Incorporate paperless billing enrollment and senior-citizen status as supporting variables within predictive churn models rather than treating them as primary indicators of churn risk.

# Numerical Hypothesis Testing

While the previous analyses examined relationships between categorical variables and customer churn using the Chi-Square Test of Independence, this section evaluates whether customers who churn differ significantly from those who remain with respect to continuous numerical variables.

Three customer metrics are analyzed:

- Monthly Charges
- Total Charges
- Customer Tenure

For each variable, the distribution of customers who churned is compared with customers who did not churn. Prior to selecting the appropriate statistical test, assumptions of normality and homogeneity of variance are evaluated. When these assumptions are satisfied, an independent samples t-test is performed. Otherwise, the non-parametric Mann–Whitney U test is used.

In addition to statistical significance, effect sizes are reported to assess the practical importance of observed differences. This ensures that conclusions are based on both statistical evidence and business relevance.

## Statistical Framework

For each numerical variable, the following analytical workflow is applied:

1. Compare the distributions of customers who churned and customers who did not churn.
2. Assess normality using the Shapiro–Wilk test.
3. Evaluate homogeneity of variance using Levene's Test.
4. Select the appropriate hypothesis test:
   - Independent Samples t-test if assumptions are satisfied.
   - Mann–Whitney U Test if assumptions are violated.
5. Calculate an appropriate effect size.
6. Interpret both statistical significance and practical significance in a business context.

The significance level for all hypothesis tests is α = 0.05.

## Helper Functions for Numerical Hypothesis Testing

The following reusable functions evaluate distributional assumptions, select an appropriate two-group hypothesis test, and calculate an effect size for each numerical variable.

Because large samples can cause formal normality tests to detect minor departures from normality, the Shapiro–Wilk results are interpreted alongside descriptive statistics and distribution plots. When both groups are approximately normal, either Student's or Welch's independent-samples t-test is used depending on Levene's test. When normality is not supported, the Mann–Whitney U test is used.

Effect sizes are reported as Cohen's *d* for t-tests and rank-biserial correlation for Mann–Whitney U tests.

In [369]:
import numpy as np
import pandas as pd

from scipy.stats import (
    levene,
    mannwhitneyu,
    shapiro,
    ttest_ind,
)

In [370]:
def cohens_d(group_1, group_2):
    """
    Calculate Cohen's d for two independent groups.

    A positive value indicates that group_1 has a higher mean.
    A negative value indicates that group_1 has a lower mean.
    """
    group_1 = np.asarray(group_1, dtype=float)
    group_2 = np.asarray(group_2, dtype=float)

    n1 = len(group_1)
    n2 = len(group_2)

    if n1 < 2 or n2 < 2:
        return np.nan

    variance_1 = np.var(group_1, ddof=1)
    variance_2 = np.var(group_2, ddof=1)

    pooled_variance = (
        ((n1 - 1) * variance_1) +
        ((n2 - 1) * variance_2)
    ) / (n1 + n2 - 2)

    if pooled_variance == 0:
        return 0.0

    return (
        np.mean(group_1) - np.mean(group_2)
    ) / np.sqrt(pooled_variance)


def rank_biserial_correlation(u_statistic, n1, n2):
    """
    Calculate rank-biserial correlation from the Mann–Whitney U statistic.

    The sign reflects the direction for group_1 relative to group_2.
    """
    if n1 == 0 or n2 == 0:
        return np.nan

    return (2 * u_statistic) / (n1 * n2) - 1

In [371]:
def interpret_effect_size(effect_size):
    """
    Interpret the absolute magnitude of Cohen's d or
    rank-biserial correlation.
    """
    if pd.isna(effect_size):
        return "Unavailable"

    magnitude = abs(effect_size)

    if magnitude < 0.10:
        return "Negligible"
    elif magnitude < 0.30:
        return "Small"
    elif magnitude < 0.50:
        return "Moderate"
    else:
        return "Large"

Main numerical-analysis function

In [372]:
def numerical_analysis(
    df,
    numerical_column,
    churn_column="churn",
    churned_label="Yes",
    retained_label="No",
    alpha=0.05,
    shapiro_max_n=5000,
):
    """
    Compare a numerical variable between churned and retained customers.

    Workflow:
    1. Convert the selected variable to numeric.
    2. Separate churned and retained customers.
    3. Evaluate normality using the Shapiro–Wilk test.
    4. Evaluate equality of variance using Levene's test.
    5. Select Student's t-test, Welch's t-test, or Mann–Whitney U.
    6. Calculate the corresponding effect size.
    """

    required_columns = {numerical_column, churn_column}
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    analysis_data = df[
        [numerical_column, churn_column]
    ].copy()

    analysis_data[numerical_column] = pd.to_numeric(
        analysis_data[numerical_column],
        errors="coerce",
    )

    analysis_data = analysis_data.dropna(
        subset=[numerical_column, churn_column]
    )

    churned = analysis_data.loc[
        analysis_data[churn_column] == churned_label,
        numerical_column,
    ].astype(float)

    retained = analysis_data.loc[
        analysis_data[churn_column] == retained_label,
        numerical_column,
    ].astype(float)

    if len(churned) < 3 or len(retained) < 3:
        raise ValueError(
            "Each churn group must contain at least three valid observations."
        )

    # Shapiro–Wilk can be unnecessarily expensive and highly sensitive
    # with large samples, so reproducible samples are used when needed.
    churned_shapiro_sample = churned.sample(
        n=min(len(churned), shapiro_max_n),
        random_state=42,
    )

    retained_shapiro_sample = retained.sample(
        n=min(len(retained), shapiro_max_n),
        random_state=42,
    )

    churned_shapiro_stat, churned_shapiro_p = shapiro(
        churned_shapiro_sample
    )

    retained_shapiro_stat, retained_shapiro_p = shapiro(
        retained_shapiro_sample
    )

    churned_normal = churned_shapiro_p >= alpha
    retained_normal = retained_shapiro_p >= alpha
    normality_satisfied = churned_normal and retained_normal

    levene_stat, levene_p = levene(
        churned,
        retained,
        center="median",
    )

    equal_variance = levene_p >= alpha

    if normality_satisfied:
        if equal_variance:
            test_name = "Independent Samples t-test"
            test_statistic, p_value = ttest_ind(
                churned,
                retained,
                equal_var=True,
                nan_policy="omit",
            )
        else:
            test_name = "Welch's Independent Samples t-test"
            test_statistic, p_value = ttest_ind(
                churned,
                retained,
                equal_var=False,
                nan_policy="omit",
            )

        effect_size = cohens_d(churned, retained)
        effect_size_name = "Cohen's d"

    else:
        test_name = "Mann–Whitney U Test"
        test_statistic, p_value = mannwhitneyu(
            churned,
            retained,
            alternative="two-sided",
            method="auto",
        )

        effect_size = rank_biserial_correlation(
            test_statistic,
            len(churned),
            len(retained),
        )

        effect_size_name = "Rank-Biserial Correlation"

    return {
        "variable": numerical_column,
        "churned": churned,
        "retained": retained,

        "churned_n": len(churned),
        "retained_n": len(retained),

        "churned_mean": churned.mean(),
        "retained_mean": retained.mean(),
        "mean_difference": churned.mean() - retained.mean(),

        "churned_median": churned.median(),
        "retained_median": retained.median(),
        "median_difference": churned.median() - retained.median(),

        "churned_std": churned.std(ddof=1),
        "retained_std": retained.std(ddof=1),

        "churned_shapiro_statistic": churned_shapiro_stat,
        "churned_shapiro_p_value": churned_shapiro_p,
        "retained_shapiro_statistic": retained_shapiro_stat,
        "retained_shapiro_p_value": retained_shapiro_p,

        "churned_normal": churned_normal,
        "retained_normal": retained_normal,
        "normality_satisfied": normality_satisfied,

        "levene_statistic": levene_stat,
        "levene_p_value": levene_p,
        "equal_variance": equal_variance,

        "test_name": test_name,
        "test_statistic": test_statistic,
        "p_value": p_value,

        "effect_size_name": effect_size_name,
        "effect_size": effect_size,
        "effect_strength": interpret_effect_size(effect_size),

        "significant": p_value < alpha,
        "alpha": alpha,
    }

Reusable summary-table function

In [373]:
def numerical_summary_table(results):
    """
    Create a standardized stakeholder-friendly summary table.
    """
    p_value_display = (
        "< 0.001"
        if results["p_value"] < 0.001
        else f"{results['p_value']:.4f}"
    )

    return pd.DataFrame({
        "Statistic": [
            "Selected Test",
            "Churned Customers",
            "Retained Customers",
            "Churned Mean",
            "Retained Mean",
            "Mean Difference",
            "Churned Median",
            "Retained Median",
            "Test Statistic",
            "P-Value",
            "Effect Size Measure",
            "Effect Size",
            "Effect Strength",
            "Normality Satisfied",
            "Equal Variance",
            "Statistically Significant",
        ],
        "Value": [
            results["test_name"],
            results["churned_n"],
            results["retained_n"],
            round(results["churned_mean"], 2),
            round(results["retained_mean"], 2),
            round(results["mean_difference"], 2),
            round(results["churned_median"], 2),
            round(results["retained_median"], 2),
            round(results["test_statistic"], 3),
            p_value_display,
            results["effect_size_name"],
            round(results["effect_size"], 3),
            results["effect_strength"],
            results["normality_satisfied"],
            results["equal_variance"],
            results["significant"],
        ],
    })

Reusable assumptions table

In [374]:
def numerical_assumptions_table(results):
    """
    Display the formal assumption-test results.
    """
    return pd.DataFrame({
        "Assumption Test": [
            "Shapiro–Wilk: Churned",
            "Shapiro–Wilk: Retained",
            "Levene's Test",
        ],
        "Statistic": [
            results["churned_shapiro_statistic"],
            results["retained_shapiro_statistic"],
            results["levene_statistic"],
        ],
        "P-Value": [
            results["churned_shapiro_p_value"],
            results["retained_shapiro_p_value"],
            results["levene_p_value"],
        ],
        "Assumption Satisfied": [
            results["churned_normal"],
            results["retained_normal"],
            results["equal_variance"],
        ],
    }).round({
        "Statistic": 4,
        "P-Value": 4,
    })

In [375]:
analysis_df["churn"].value_counts(dropna=False)

churn
0    5174
1    1869
Name: count, dtype: int64

In [376]:
def cohens_d(group_1, group_2):
    """
    Calculate Cohen's d for two independent groups.
    """
    group_1 = np.asarray(group_1, dtype=float)
    group_2 = np.asarray(group_2, dtype=float)

    n1 = len(group_1)
    n2 = len(group_2)

    if n1 < 2 or n2 < 2:
        return np.nan

    variance_1 = np.var(group_1, ddof=1)
    variance_2 = np.var(group_2, ddof=1)

    pooled_variance = (
        ((n1 - 1) * variance_1)
        + ((n2 - 1) * variance_2)
    ) / (n1 + n2 - 2)

    if pooled_variance == 0:
        return 0.0

    return (
        np.mean(group_1) - np.mean(group_2)
    ) / np.sqrt(pooled_variance)


def rank_biserial_correlation(u_statistic, n1, n2):
    """
    Calculate rank-biserial correlation from a Mann-Whitney U statistic.
    """
    if n1 == 0 or n2 == 0:
        return np.nan

    return (2 * u_statistic) / (n1 * n2) - 1


def interpret_effect_size(effect_size):
    """
    Interpret the absolute magnitude of an effect size.
    """
    if pd.isna(effect_size):
        return "Unavailable"

    magnitude = abs(effect_size)

    if magnitude < 0.10:
        return "Negligible"
    elif magnitude < 0.30:
        return "Small"
    elif magnitude < 0.50:
        return "Moderate"
    else:
        return "Large"

In [377]:
def numerical_analysis(
    df,
    numerical_column,
    churn_column="churn",
    churned_label="1",
    retained_label="0",
    alpha=0.05,
    shapiro_max_n=5000,
):
    """
    Compare a numerical variable between churned and retained customers.

    Workflow:
    1. Validate the required columns.
    2. Convert the numerical column to numeric.
    3. Remove missing values.
    4. Separate churned and retained customers.
    5. Evaluate normality using the Shapiro-Wilk test.
    6. Evaluate equality of variance using Levene's test.
    7. Select a t-test or Mann-Whitney U test.
    8. Calculate the appropriate effect size.
    """

    required_columns = {numerical_column, churn_column}
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Create a clean analytical subset.
    analysis_data = df[
        [numerical_column, churn_column]
    ].copy()

    # Ensure that the selected variable is numeric.
    analysis_data[numerical_column] = pd.to_numeric(
        analysis_data[numerical_column],
        errors="coerce",
    )

    # Remove rows with missing values in either required column.
    analysis_data = analysis_data.dropna(
        subset=[numerical_column, churn_column]
    )

    # Separate churned and retained customers.
    churned = analysis_data.loc[
        analysis_data[churn_column] == churned_label,
        numerical_column,
    ].astype(float)

    retained = analysis_data.loc[
        analysis_data[churn_column] == retained_label,
        numerical_column,
    ].astype(float)

    # Provide a useful error message if labels do not match.
    if len(churned) < 3 or len(retained) < 3:
        available_labels = analysis_data[
            churn_column
        ].value_counts(dropna=False).to_dict()

        raise ValueError(
            "Unable to create valid churn groups. "
            f"Expected churned_label={churned_label!r} and "
            f"retained_label={retained_label!r}. "
            f"Available values and counts: {available_labels}"
        )

    # Shapiro-Wilk is highly sensitive with large datasets.
    # Use a reproducible sample if a group exceeds the selected limit.
    churned_shapiro_sample = churned.sample(
        n=min(len(churned), shapiro_max_n),
        random_state=42,
    )

    retained_shapiro_sample = retained.sample(
        n=min(len(retained), shapiro_max_n),
        random_state=42,
    )

    churned_shapiro_stat, churned_shapiro_p = shapiro(
        churned_shapiro_sample
    )

    retained_shapiro_stat, retained_shapiro_p = shapiro(
        retained_shapiro_sample
    )

    churned_normal = churned_shapiro_p >= alpha
    retained_normal = retained_shapiro_p >= alpha
    normality_satisfied = churned_normal and retained_normal

    # Test homogeneity of variance.
    levene_stat, levene_p = levene(
        churned,
        retained,
        center="median",
    )

    equal_variance = levene_p >= alpha

    # Select the statistical test.
    if normality_satisfied:
        if equal_variance:
            test_name = "Independent Samples t-test"

            test_statistic, p_value = ttest_ind(
                churned,
                retained,
                equal_var=True,
                nan_policy="omit",
            )
        else:
            test_name = "Welch's Independent Samples t-test"

            test_statistic, p_value = ttest_ind(
                churned,
                retained,
                equal_var=False,
                nan_policy="omit",
            )

        effect_size = cohens_d(
            churned,
            retained,
        )

        effect_size_name = "Cohen's d"

    else:
        test_name = "Mann-Whitney U Test"

        test_statistic, p_value = mannwhitneyu(
            churned,
            retained,
            alternative="two-sided",
            method="auto",
        )

        effect_size = rank_biserial_correlation(
            test_statistic,
            len(churned),
            len(retained),
        )

        effect_size_name = "Rank-Biserial Correlation"

    return {
        "variable": numerical_column,

        "churned": churned,
        "retained": retained,

        "churned_n": len(churned),
        "retained_n": len(retained),

        "churned_mean": churned.mean(),
        "retained_mean": retained.mean(),
        "mean_difference": churned.mean() - retained.mean(),

        "churned_median": churned.median(),
        "retained_median": retained.median(),
        "median_difference": churned.median() - retained.median(),

        "churned_std": churned.std(ddof=1),
        "retained_std": retained.std(ddof=1),

        "churned_shapiro_statistic": churned_shapiro_stat,
        "churned_shapiro_p_value": churned_shapiro_p,

        "retained_shapiro_statistic": retained_shapiro_stat,
        "retained_shapiro_p_value": retained_shapiro_p,

        "churned_normal": churned_normal,
        "retained_normal": retained_normal,
        "normality_satisfied": normality_satisfied,

        "levene_statistic": levene_stat,
        "levene_p_value": levene_p,
        "equal_variance": equal_variance,

        "test_name": test_name,
        "test_statistic": test_statistic,
        "p_value": p_value,

        "effect_size_name": effect_size_name,
        "effect_size": effect_size,
        "effect_strength": interpret_effect_size(effect_size),

        "significant": p_value < alpha,
        "alpha": alpha,
    }

In [378]:
monthly_results = numerical_analysis(
    analysis_df,
    "monthly_charges",
)

display(numerical_assumptions_table(monthly_results))
display(numerical_summary_table(monthly_results))

,Assumption Test,Statistic,P-Value,Assumption Satisfied
0,Shapiro–Wilk: Churned,0.9284,0.0000,False
1,Shapiro–Wilk: Retained,0.9129,0.0000,False
2,Levene's Test,361.8445,0.0000,False


,Statistic,Value
0,Selected Test,Mann-Whitney U Test
1,Churned Customers,1869
2,Retained Customers,5174
3,Churned Mean,74.4400
4,Retained Mean,61.2700
5,Mean Difference,13.1800
6,Churned Median,79.6500
7,Retained Median,64.4300
8,Test Statistic,"6,003,125.5000"
9,P-Value,< 0.001


### Assumption Assessment

The assumptions required for a parametric independent samples t-test were evaluated prior to hypothesis testing.

The Shapiro–Wilk test indicated that the distributions of monthly charges for both churned and retained customers deviated significantly from normality (p < 0.001). In addition, Levene's Test indicated unequal variances between the two groups (p < 0.001).

Because both assumptions were violated, the Mann–Whitney U Test was selected as the appropriate non-parametric alternative for comparing monthly charges between churned and retained customers.

In [379]:
display(numerical_summary_table(monthly_results))

,Statistic,Value
0,Selected Test,Mann-Whitney U Test
1,Churned Customers,1869
2,Retained Customers,5174
3,Churned Mean,74.4400
4,Retained Mean,61.2700
5,Mean Difference,13.1800
6,Churned Median,79.6500
7,Retained Median,64.4300
8,Test Statistic,"6,003,125.5000"
9,P-Value,< 0.001


### Statistical Interpretation

The Mann–Whitney U Test was conducted to compare monthly charges between customers who churned and those who remained with the company. The test identified a statistically significant difference between the two groups (U = 6,003,125.50, p < 0.001). Therefore, the null hypothesis is rejected, indicating that monthly charges differ significantly between churned and retained customers.

A non-parametric test was selected because the assumptions required for an independent samples t-test were not satisfied. Both groups deviated significantly from normality according to the Shapiro–Wilk test, and Levene's Test indicated unequal variances.

The effect size, measured using the rank-biserial correlation (r = 0.242), indicates a **small practical effect**. Although the difference in monthly charges is statistically significant, the magnitude of the difference is modest, suggesting that monthly charges should be considered alongside other customer characteristics when evaluating churn risk.

Consistent with the exploratory data analysis, customers who churned had higher monthly charges (mean = \$74.44; median = \$79.65) than customers who remained with the company (mean = \$61.27; median = \$64.43).

### Business Insight

Customers who churned generally paid higher monthly charges than customers who remained with the company, suggesting that higher recurring costs are associated with an increased likelihood of customer attrition.

However, the relatively small effect size indicates that monthly charges alone are unlikely to fully explain customer churn. Pricing should therefore be considered as one component of a broader customer risk profile rather than an independent predictor of churn.

Organizations may benefit from monitoring customers with higher monthly charges, particularly when these customers also exhibit stronger risk factors such as month-to-month contracts, limited support-service adoption, or fiber optic internet service. Combining pricing information with other operational characteristics can support more effective retention strategies and targeted customer interventions.

### Conclusion

The analysis demonstrates that monthly charges differ significantly between churned and retained customers. Customers who churned tended to pay higher monthly charges; however, the practical effect of this difference is relatively small. These findings suggest that pricing contributes to customer churn but should be interpreted alongside stronger operational factors identified in the categorical analyses.

## Research Question 8

### Do total charges differ between customers who churned and customers who remained?

Total charges represent the cumulative amount paid by a customer throughout their relationship with the company. This analysis evaluates whether total charges differ significantly between customers who churned and customers who remained.

### Hypotheses

**Null Hypothesis (H₀)**

The distribution of total charges is the same for churned and retained customers.

**Alternative Hypothesis (H₁)**

The distribution of total charges differs between churned and retained customers.

### Statistical Test

The appropriate statistical test is selected automatically based on the results of the assumption assessments.

### Effect Size

Cohen's *d* (parametric tests) or Rank-Biserial Correlation (non-parametric tests)

In [384]:
total_results = numerical_analysis(
    analysis_df,
    "total_charges",
)

display(numerical_assumptions_table(total_results))
display(numerical_summary_table(total_results))

,Assumption Test,Statistic,P-Value,Assumption Satisfied
0,Shapiro–Wilk: Churned,0.7787,0.0000,False
1,Shapiro–Wilk: Retained,0.8827,0.0000,False
2,Levene's Test,166.3004,0.0000,False


,Statistic,Value
0,Selected Test,Mann-Whitney U Test
1,Churned Customers,1869
2,Retained Customers,5174
3,Churned Mean,"1,531.8000"
4,Retained Mean,"2,549.9100"
5,Mean Difference,"-1,018.1200"
6,Churned Median,703.5500
7,Retained Median,"1,679.5200"
8,Test Statistic,"3,381,224.0000"
9,P-Value,< 0.001


### Assumption Assessment

The assumptions required for an independent samples t-test were evaluated prior to hypothesis testing.

The Shapiro–Wilk test indicated that the distributions of total charges for both churned and retained customers deviated significantly from normality (p < 0.001). In addition, Levene's Test indicated unequal variances between the two groups (p < 0.001).

Because both assumptions were violated, the Mann–Whitney U Test was selected as the appropriate non-parametric method for comparing total charges between churned and retained customers.

### Statistical Interpretation

The Mann–Whitney U Test identified a statistically significant difference in total charges between customers who churned and those who remained with the company (U = 3,381,224.00, p < 0.001). Therefore, the null hypothesis is rejected, indicating that total charges differ significantly between the two customer groups.

A non-parametric test was selected because the assumptions of normality and equal variances were not satisfied. Both customer groups exhibited significant departures from normality according to the Shapiro–Wilk test, and Levene's Test indicated unequal variances.

The effect size, measured using the rank-biserial correlation (|r| = 0.301), indicates a **moderate practical effect**. This suggests that the observed difference in total charges is not only statistically significant but also meaningful from a business perspective.

Consistent with the exploratory data analysis, retained customers accumulated substantially higher total charges (mean = \$2,549.91; median = \$1,679.52) than customers who churned (mean = \$1,531.80; median = \$703.55).

### Business Insight

Customers who remained with the company accumulated significantly higher total charges than customers who churned. This finding reflects the fact that customers who maintain longer relationships with the company naturally generate greater lifetime revenue.

The moderate effect size indicates that total charges are a meaningful indicator of customer retention. However, total charges should not be interpreted as a direct cause of customer loyalty, as they are strongly influenced by customer tenure.

Instead, total charges are best viewed as a valuable business metric that reflects customer lifetime value. When combined with operational factors such as contract type, support-service adoption, and customer tenure, total charges can improve customer segmentation and predictive churn models by identifying customers at elevated risk of attrition.

### Conclusion

The analysis demonstrates that total charges differ significantly between churned and retained customers. Retained customers accumulated substantially higher lifetime spending than customers who churned, reflecting the longer duration of their customer relationships. The moderate effect size indicates that total charges provide meaningful information for understanding customer retention, although the variable should be interpreted in conjunction with customer tenure and other operational factors.

## Research Question 9

### Does customer tenure differ between customers who churned and customers who remained?

Customer tenure measures the length of time a customer has maintained their service subscription. This analysis evaluates whether tenure differs significantly between customers who churned and customers who remained.

### Hypotheses

**Null Hypothesis (H₀)**

The distribution of customer tenure is the same for churned and retained customers.

**Alternative Hypothesis (H₁)**

The distribution of customer tenure differs between churned and retained customers.

### Statistical Test

The appropriate statistical test is selected automatically based on the results of the assumption assessments.

### Effect Size

Cohen's *d* (parametric tests) or Rank-Biserial Correlation (non-parametric tests)

In [381]:
tenure_results = numerical_analysis(
    analysis_df,
    "tenure",
)

display(numerical_assumptions_table(tenure_results))
display(numerical_summary_table(tenure_results))

,Assumption Test,Statistic,P-Value,Assumption Satisfied
0,Shapiro–Wilk: Churned,0.8200,0.0000,False
1,Shapiro–Wilk: Retained,0.9182,0.0000,False
2,Levene's Test,417.1696,0.0000,False


,Statistic,Value
0,Selected Test,Mann-Whitney U Test
1,Churned Customers,1869
2,Retained Customers,5174
3,Churned Mean,17.9800
4,Retained Mean,37.5700
5,Mean Difference,-19.5900
6,Churned Median,10.0000
7,Retained Median,38.0000
8,Test Statistic,"2,515,538.0000"
9,P-Value,< 0.001


### Assumption Assessment

The assumptions required for an independent samples t-test were evaluated prior to hypothesis testing.

The Shapiro–Wilk test indicated that the distributions of customer tenure for both churned and retained customers deviated significantly from normality (p < 0.001). In addition, Levene's Test indicated unequal variances between the two groups (p < 0.001).

Because both assumptions were violated, the Mann–Whitney U Test was selected as the appropriate non-parametric method for comparing customer tenure between churned and retained customers.

### Statistical Interpretation

The Mann–Whitney U Test identified a statistically significant difference in customer tenure between customers who churned and those who remained with the company (U = 2,515,538.00, p < 0.001). Therefore, the null hypothesis is rejected, indicating that customer tenure differs significantly between the two customer groups.

A non-parametric test was selected because the assumptions of normality and equal variances were not satisfied. Both customer groups exhibited significant departures from normality according to the Shapiro–Wilk test, and Levene's Test indicated unequal variances.

The effect size, measured using the rank-biserial correlation (|r| = 0.480), indicates a **moderate practical effect** and represents the strongest numerical relationship identified in this study.

Consistent with the exploratory data analysis, customers who remained with the company had substantially longer tenures (mean = 37.57 months; median = 38 months) than customers who churned (mean = 17.98 months; median = 10 months).

### Business Insight

Customer tenure represents the strongest numerical indicator of customer churn identified in this analysis. Customers who churn generally leave much earlier in their relationship with the company than customers who remain.

This finding highlights the importance of the early stages of the customer lifecycle. Organizations should prioritize onboarding, proactive customer support, and engagement initiatives during the first months of a customer's subscription to improve long-term retention.

Although tenure itself cannot be directly influenced, identifying customers with shorter subscription histories can help organizations target retention programs before customers become disengaged. When combined with operational factors such as contract type and support-service adoption, tenure provides valuable information for identifying customers at elevated risk of churn.

### Conclusion

The analysis demonstrates that customer tenure differs significantly between churned and retained customers. Customers who remained with the company maintained substantially longer relationships than customers who churned. Among the numerical variables examined, customer tenure exhibited the strongest practical association with churn and should be considered a key feature in predictive churn modeling and customer retention strategies.

# Summary of Numerical Analyses

The numerical hypothesis tests evaluated whether customers who churned differed significantly from retained customers with respect to monthly charges, total charges, and customer tenure.

All three variables demonstrated statistically significant differences between customer groups (p < 0.001). However, the practical importance of these differences varied according to their effect sizes. The results are summarized below.

In [382]:
numerical_summary = pd.DataFrame({
    "Rank": [1, 2, 3],
    "Variable": [
        "Customer Tenure",
        "Total Charges",
        "Monthly Charges"
    ],
    "Selected Test": [
        "Mann-Whitney U",
        "Mann-Whitney U",
        "Mann-Whitney U"
    ],
    "Higher-Risk Pattern": [
        "Shorter Tenure",
        "Lower Lifetime Spending",
        "Higher Monthly Charges"
    ],
    "Effect Size": [
        0.480,
        0.301,
        0.242
    ],
    "Effect Strength": [
        "Moderate",
        "Moderate",
        "Small"
    ],
    "Business Priority": [
        "Very High",
        "High",
        "Medium"
    ]
})

display(numerical_summary)

,Rank,Variable,Selected Test,Higher-Risk Pattern,Effect Size,Effect Strength,Business Priority
0,1,Customer Tenure,Mann-Whitney U,Shorter Tenure,0.4800,Moderate,Very High
1,2,Total Charges,Mann-Whitney U,Lower Lifetime Spending,0.3010,Moderate,High
2,3,Monthly Charges,Mann-Whitney U,Higher Monthly Charges,0.2420,Small,Medium


## Key Findings

Among the numerical variables examined, **customer tenure** exhibited the strongest practical association with churn. Customers who churned maintained substantially shorter relationships with the company than retained customers, highlighting the importance of the early stages of the customer lifecycle.

**Total charges** also demonstrated a meaningful difference between customer groups. Because total charges naturally accumulate over time, this finding complements the tenure analysis by illustrating the financial value associated with long-term customer retention.

Although **monthly charges** differed significantly between churned and retained customers, the relatively small effect size suggests that pricing alone is not a primary driver of customer churn. Monthly charges should therefore be interpreted alongside stronger operational and behavioral factors.

# Executive Summary

This project investigated customer churn using exploratory data analysis and statistical hypothesis testing on the IBM Telco Customer Churn dataset. The objective was to identify customer characteristics most strongly associated with churn and translate those findings into actionable business recommendations.

The exploratory analysis identified clear differences in churn rates across customer segments. These observations were subsequently validated using six Chi-Square Tests of Independence for categorical variables and three non-parametric Mann–Whitney U Tests for numerical variables.

Among the categorical variables, **contract type** exhibited the strongest association with churn, followed by **support-service adoption**, **internet service type**, and **payment method**. Month-to-month customers experienced substantially higher churn than customers with longer-term contracts, while customers without support services exhibited the highest observed churn rate in the dataset. These findings suggest that customer engagement and service adoption represent important opportunities for reducing customer attrition.

Among the numerical variables, **customer tenure** demonstrated the strongest practical relationship with churn. Customers who churned maintained significantly shorter relationships with the company than retained customers. Total charges also differed meaningfully between customer groups, reflecting the cumulative value of long-term customer retention. Monthly charges were significantly higher among customers who churned; however, the relatively small effect size indicates that pricing should be interpreted alongside other customer characteristics rather than in isolation.

Overall, the analyses indicate that customer churn is influenced more strongly by operational and behavioral characteristics than by demographic attributes. Contract structure, support-service adoption, internet service type, payment method, and customer tenure consistently demonstrated the greatest practical importance. These findings provide a statistically validated foundation for customer segmentation, predictive churn modeling, and targeted retention strategies.

# Final Business Recommendations

Based on the statistical analyses conducted in this project, the following recommendations are proposed:

1. Prioritize retention initiatives for customers on month-to-month contracts through renewal incentives and personalized offers.
2. Increase adoption of support services such as online security, online backup, device protection, and technical support through onboarding programs and bundled service offerings.
3. Investigate the elevated churn observed among fiber optic customers to identify potential pricing, service quality, or competitive issues.
4. Encourage enrollment in automatic payment methods to reduce the elevated churn associated with electronic check payments.
5. Focus customer success efforts during the early stages of the customer lifecycle, as shorter tenure demonstrated the strongest numerical association with churn.
6. Develop predictive churn models that integrate operational, behavioral, and numerical customer characteristics rather than relying on any single variable.

Collectively, these recommendations provide a data-driven framework for improving customer retention while maximizing the long-term value of customer relationships.